In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [50]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
    
data_transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

train_data=datasets.ImageFolder(root="data/train",transform=data_transform)
test_data=datasets.ImageFolder(root="data/test",transform=data_transform)
valid_data=datasets.ImageFolder(root="data/validation",transform=data_transform)




In [ ]:
train_loader=DataLoader(train_data,batch_size=32,shuffle=True,num_workers=2)
test_loader=DataLoader(test_data,batch_size=32,shuffle=False,num_workers=2)
valid_loader=DataLoader(valid_data,batch_size=32,shuffle=False,num_workers=2)

In [52]:
class_names=train_data.classes

In [ ]:
from torchvision.models import resnet18
import torch.nn as nn

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=resnet18(weights="IMAGENET1K_V1")
model.fc=nn.Linear(512,9)
model=model.to(device)


In [ ]:
import torch.optim as optim

loss=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

epochs=15   
for epoch in range(epochs):
    model.train()
    running_loss=0

    for img,label in train_loader:
        img,label=img.to(device),label.to(device)
        optimizer.zero_grad()
        output=model(img)
        losses=loss(output,label)
        losses.backward()
        optimizer.step()
        running_loss+=losses.item()

    print(f"Epoch: {epoch+1}, loss: {running_loss/len(train_loader)}")



Epoch: 1, loss: 0.46509973081083245
Epoch: 2, loss: 0.3077760301652502
Epoch: 3, loss: 0.25247691141675566
Epoch: 4, loss: 0.20624010172555518
Epoch: 5, loss: 0.17152609617953174
Epoch: 6, loss: 0.13727121616566115
Epoch: 7, loss: 0.10660076866580347
Epoch: 8, loss: 0.08407121287023298
Epoch: 9, loss: 0.07183120622368952
Epoch: 10, loss: 0.05620633302788604
Epoch: 11, loss: 0.05129254364209438
Epoch: 12, loss: 0.04157385607745436
Epoch: 13, loss: 0.04021665252620466
Epoch: 14, loss: 0.033314385330329155
Epoch: 15, loss: 0.03162140763663961


Evaluation on test set

In [55]:
model.eval()
correct=0
tot=0

with torch.no_grad():
    for img,label in test_loader:
        img,label=img.to(device),label.to(device)
        output=model(img)
        _,predicted=torch.max(output,1)
        tot+=label.size(0)
        correct+=(predicted==label).sum().item()

print(f"Test data accuracy: {100*(correct/tot)}%")

Test data accuracy: 96.71013633669236%


Evaluation of Validation data

In [56]:
model.eval()
correct=0
tot=0

with torch.no_grad():
    for img,label in valid_loader:
        img,label=img.to(device),label.to(device)
        output=model(img)
        _,predicted=torch.max(output,1)
        tot+=label.size(0)
        correct+=(predicted==label).sum().item()

print(f"Validation data accuracy: {100*(correct/tot)}%")


Validation data accuracy: 96.6688874083944%


In [57]:
# Save only the weights (Standard PyTorch practice)
torch.save(model.state_dict(), 'wafer_resnet18_96acc.pth')
print("Model saved successfully!")

Model saved successfully!
